In [2]:
import pandas as pd

In [3]:
df = pd.DataFrame({"A":[1,2], "B":[3,4]})
df

,A,B
0,1,3
1,2,4


In [1]:
import sys

print(sys.executable)

c:\Users\lizcr\OneDrive\Documents\MSc\Project\msc_project\.venv\Scripts\python.exe


In [2]:
import copy, math, os, pickle, time, pandas as pd, numpy as np, scipy.stats as ss

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, accuracy_score, f1_score

import torch, torch.utils.data as utils, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.autograd import Variable
from torch.nn.parameter import Parameter

In [ ]:
#GAP_TIME          = 6  # In hours.  I don't need this as there is already a gap between end of data and predicted outcome (at 7 days)
WINDOW_SIZE       = 49 # In hours.  49 hours as I have 24 hours prior to ICU inttime, 24 hours in ICU.  
SEED              = 1
ID_COLS           = ['subject_id', 'hadm_id', 'icustay_id']
ID_COLS_HOURLY = ID_COLS + ['hours_in']
TESTING = True          # set False for the full cohort run

np.random.seed(SEED)
torch.manual_seed(SEED)

In [4]:
class DictDist():
    def __init__(self, dict_of_rvs): self.dict_of_rvs = dict_of_rvs
    def rvs(self, n):
        a = {k: v.rvs(n) for k, v in self.dict_of_rvs.items()}
        out = []
        for i in range(n): out.append({k: vs[i] for k, vs in a.items()})
        return out
    
class Choice():
    def __init__(self, options): self.options = options
    def rvs(self, n): return [self.options[i] for i in ss.randint(0, len(self.options)).rvs(n)]

In [5]:
import getpass
from sqlalchemy import create_engine

pg_user = 'postgres'      # same value you use to connect via psql
pg_host = 'localhost'          # or wherever your Postgres server is
pg_port = 5432
pg_dbname = 'mimiciv'

pg_password = getpass.getpass('Postgres password: ')

engine = create_engine(
    f'postgresql+psycopg2://{pg_user}:{pg_password}@{pg_host}:{pg_port}/{pg_dbname}'
)

In [6]:
df = pd.read_sql("SELECT current_database();", engine)
print(df)

  current_database
0          mimiciv


In [7]:
pd.read_sql('SELECT 1', engine)

,?column?
0,1


In [16]:
%%time

hourly_table = 'msc_project.sample_hourly_data' if TESTING else 'msc_project.hourly_data'
statics_table = 'msc_project.sample_allpatients' if TESTING else 'msc_project.allpatients'

hourly_query = f"SELECT * FROM {hourly_table}"
statics_query = f"SELECT * FROM {statics_table}"

data_full_lvl2 = (
    pd.read_sql(hourly_query, engine)
    .set_index(ID_COLS_HOURLY)
    .drop(columns='hour_end')   # dropping hour_end as it is redundant - hours_in tracks time since admission.
)

statics = pd.read_sql(statics_query, engine).set_index(ID_COLS)

CPU times: total: 78.1 ms
Wall time: 184 ms


In [17]:
data_full_lvl2.head()

heart_rate    sbp   dbp   mbp  \
subject_id hadm_id  icustay_id hours_in                                  
12894275   20050533 30397733   24              88.0  119.0  63.0  74.0   
                               23              85.0  114.0  63.0  74.0   
                               22              99.0  117.0  67.0  78.0   
                               21              90.0  116.0  54.0  68.0   
                               20              89.0  120.0  68.5  76.5   

                                         sbp_ni  dbp_ni  mbp_ni  temperature  \
subject_id hadm_id  icustay_id hours_in                                        
12894275   20050533 30397733   24         119.0    63.0    74.0          NaN   
                               23         114.0    63.0    74.0          NaN   
                               22         117.0    67.0    78.0          NaN   
                               21         116.0    54.0    68.0        37.17   
                               20         120.0    68.5    76.5          NaN   

                                         spo2  glucose_vital  ...   gcs  \
subject_id hadm_id  icustay_id hours_in                       ...         
12894275   20050533 30397733   24        98.0            NaN  ...   NaN   
                               23        99.0            NaN  ...   NaN   
                               22        98.0            NaN  ...   NaN   
                               21        98.0            NaN  ...  15.0   
                               20        99.0            NaN  ...   NaN   

                                        hematocrit hemoglobin  mch  mchc  mcv  \
subject_id hadm_id  icustay_id hours_in                                         
12894275   20050533 30397733   24              NaN        NaN  NaN   NaN  NaN   
                               23              NaN        NaN  NaN   NaN  NaN   
                               22              NaN        NaN  NaN   NaN  NaN   
                               21              NaN        NaN  NaN   NaN  NaN   
                               20              NaN        NaN  NaN   NaN  NaN   

                                         platelet  rbc  rdw  wbc  
subject_id hadm_id  icustay_id hours_in                           
12894275   20050533 30397733   24             NaN  NaN  NaN  NaN  
                               23             NaN  NaN  NaN  NaN  
                               22             NaN  NaN  NaN  NaN  
                               21             NaN  NaN  NaN  NaN  
                               20             NaN  NaN  NaN  NaN  

[5 rows x 33 columns]

In [18]:
statics.head()

,,,gender,dod,admittime,dischtime,admission_type,admission_location,admission_age,race,hospital_expire_flag,los_icu
subject_id,hadm_id,icustay_id,,,,,,,,,,
15386345,20490942,32585414,M,2160-02-07,2156-09-30 16:31:00,2156-10-14 14:45:00,EW EMER.,WALK-IN/SELF REFERRAL,71,WHITE,0,1.08
18583067,25650481,31835914,M,2125-05-31,2124-12-26 16:27:00,2125-01-07 20:11:00,EW EMER.,EMERGENCY ROOM,52,WHITE,0,1.96
17122654,27905430,35106498,F,2170-10-20,2170-09-03 19:04:00,2170-09-16 14:35:00,URGENT,TRANSFER FROM HOSPITAL,83,WHITE,0,5.00
10544642,27891616,37540625,F,None,2126-06-21 13:28:00,2126-06-26 18:05:00,EW EMER.,PROCEDURE SITE,70,WHITE,0,3.88
11001569,23809683,30851202,F,2146-06-29,2146-06-20 16:29:00,2146-06-29 03:46:00,OBSERVATION ADMIT,WALK-IN/SELF REFERRAL,88,WHITE - RUSSIAN,1,6.00


In [ ]:
""""
Adapted from the MIMIC-Extract function 'simple_imputer' in mimic3benchmark.preprocessing.utils, which
deals with missing values in the hourly data. The original function takes a dataframe with a multi-index of 
(subject_id, hadm_id, icustay_id, hours_in) and columns with a multi-index of (label, LEVEL1, LEVEL2, 
Aggregation Function). The function fills in missing values for the 'mean' aggregation function using 
forward fill and the mean of the icustay, and creates a 'mask' column indicating whether the original value 
was present or not. It also calculates the time since the last measurement for each variable.

This adaptation takes a dataframe with only one column per variable for the mean per hour. This is indexed by 
the ID columns and hour. If there is no measurement in a given hour, it is supplied as NaN in the input. 

I have made a decision to forward fill the missing values, but not to fill missing values at the start with the 
mean. It seems incorrect to impute values for hours before the first measurement - I do not want to use measures from 
after the relevant time point to impute values for before the first measurement.

"""

def simple_imputer(df, id_cols=ID_COLS):

    df = df.copy()
    # NB the forward fill depends on hours_in being in order for each stay.
    df = df.sort_index()    # Therefore sort. The index has been set earlier to be the ID_COLS + ['hours_in'] for the hourly data.

    mask = df.notna().astype(float)    # see later for why we convert to float. This is the mask of whether a 
                                        # measurement was present or not.
    imputed = (
        df.groupby(level=id_cols).ffill()              # forward fills from the last measurement.
        .fillna(0)
    )

    # Planning to grouping by stay here, this will mean that the last observed time should be only within the stay, 
    # not across other stays as in the original notebook. 
    is_absent = 1 - mask                # This is now 1 if the measurement is absent, 0 if there is a measurement.
    total_hours_of_absence = is_absent.groupby(level=id_cols).cumsum()   # cumulative sum of hours with no measurement for the stay.
    absent_hours_before_last_measure = (
        total_hours_of_absence[is_absent == 0]
        .groupby(level=id_cols).ffill()   
    )   # This is the number of absent hours before the latest measurement. 
    hours_since_last_measure = (total_hours_of_absence - absent_hours_before_last_measure).fillna(100)  # 100 is beyond the max hours covered in my data.

    df_out = pd.concat(
        [imputed, mask, hours_since_last_measure],             # temporary - include all of the measures to review progress. 
        axis = 1,
        keys = ['mean', 'mask', 'hours_since_last_measure']     # rename imputed as mean to match the Wang pipeline code.
    )

    # There will need to be some renaming and restructuring here to work with later code from the Wang pipeline.
    df_out.columns.names = ['Aggregation Function', 'variable']     # need top level names
    df_out = df_out.swaplevel(0, 1, axis=1)                           # swap levels so variable is top level, aggregation next, to match Wang pipeline.
    df_out.sort_index(axis=1, inplace=True)                             # sort index to be consistent with the Wang pipeline.

    return df_out

SyntaxError: invalid syntax (3988601190.py, line 42)

In [33]:
test_output =simple_imputer(data_full_lvl2).head(50)
test_output

variable                                                 albumin               \
Aggregation Function                    hours_since_last_measure imputed mask   
subject_id hadm_id  icustay_id hours_in                                         
10164613   22813323 39738251   -24                         100.0     0.0  0.0   
                               -23                         100.0     0.0  0.0   
                               -22                         100.0     0.0  0.0   
                               -21                         100.0     0.0  0.0   
                               -20                         100.0     0.0  0.0   
                               -19                         100.0     0.0  0.0   
                               -18                         100.0     0.0  0.0   
                               -17                         100.0     0.0  0.0   
                               -16                         100.0     0.0  0.0   
                               -15                         100.0     0.0  0.0   
                               -14                         100.0     0.0  0.0   
                               -13                         100.0     0.0  0.0   
                               -12                         100.0     0.0  0.0   
                               -11                         100.0     0.0  0.0   
                               -10                         100.0     0.0  0.0   
                               -9                          100.0     0.0  0.0   
                               -8                          100.0     0.0  0.0   
                               -7                          100.0     0.0  0.0   
                               -6                          100.0     0.0  0.0   
                               -5                          100.0     0.0  0.0   
                               -4                          100.0     0.0  0.0   
                               -3                          100.0     0.0  0.0   
                               -2                          100.0     0.0  0.0   
                               -1                          100.0     0.0  0.0   
                                0                          100.0     0.0  0.0   
                                1                          100.0     0.0  0.0   
                                2                          100.0     0.0  0.0   
                                3                          100.0     0.0  0.0   
                                4                          100.0     0.0  0.0   
                                5                          100.0     0.0  0.0   
                                6                          100.0     0.0  0.0   
                                7                          100.0     0.0  0.0   
                                8                          100.0     0.0  0.0   
                                9                          100.0     0.0  0.0   
                                10                         100.0     0.0  0.0   
                                11                         100.0     0.0  0.0   
                                12                         100.0     0.0  0.0   
                                13                         100.0     0.0  0.0   
                                14                         100.0     0.0  0.0   
                                15                         100.0     0.0  0.0   
                                16                         100.0     0.0  0.0   
                                17                         100.0     0.0  0.0   
                                18                         100.0     0.0  0.0   
                                19                         100.0     0.0  0.0   
                                20                         100.0     0.0  0.0   
                                21                         100.0     0.0  0.0   
                               

In [ ]:
""""
Adapted from the MIMIC-Extract cell in the notebook 'Baselines for Mortality and LOS prediction - SKlearn.ipynb'
in mimic3benchmark.preprocessing.utils, which splits the data into train, dev and test sets and also deals 
with standardisation. 
Updates compared to the original code:
- Removed the raw dataset as I am not using it.
- limited to a single outcome, length of stay after 7 days from ICU admission.
- No truncation based on the number of hours in the ICU, as this has already been done in my SQL processing.
- No use of GAP_TIME to ensure a gap between the end of the data and the predicted outcome, as there is already a gap between the last 
time point at 24 hours post-ICU admission and the predicted outcome at 7 days post ICU admission.
"""


Ys = statics[['los_icu']].copy()  # No need to filter for max_hours > WINDOW_SIZE + GAP_TIME as this check is done in the SQL query for the hourly data. GAP_TIME is not needed as there is already a gap between the end of the data and the predicted outcome (at 7 days). 
Ys['los_7'] = Ys['los_icu'] > 7
Ys.drop(columns=['los_icu'], inplace=True)
Ys.astype(float)

lvl2 = data_full_lvl2[             # No requirement for a raw data set. No need to limit to values within WINDOW_SIZE.
                                    # to do - add a check for number of hours instead?
    (data_full_lvl2.index.get_level_values('icustay_id').isin(
        set(Ys.index.get_level_values('icustay_id'))
        )
        )
]

train_frac, dev_frac, test_frac = 0.7, 0.1, 0.2
lvl2_subj_idx, Ys_subj_idx = [df.index.get_level_values('subject_id') for df in (lvl2, Ys)] # no raw dataset.
lvl2_subjects = set(lvl2_subj_idx)
assert lvl2_subjects == set(Ys_subj_idx), "Subject ID pools differ!"

np.random.seed(SEED)
subjects, N = np.random.permutation(list(lvl2_subjects)), len(lvl2_subjects)
N_train, N_dev, N_test = int(train_frac * N), int(dev_frac * N), int(test_frac * N)
train_subj = subjects[:N_train]
dev_subj   = subjects[N_train:N_train + N_dev]
test_subj  = subjects[N_train+N_dev:]

[(lvl2_train, lvl2_dev, lvl2_test), (Ys_train, Ys_dev, Ys_test)] = [     # again no raw dataset.
    [df[df.index.get_level_values('subject_id').isin(s)] for s in (train_subj, dev_subj, test_subj)] \
    for df in (lvl2, Ys)
]

idx = pd.IndexSlice
lvl2_means, lvl2_stds = lvl2_train.loc[:, idx[:,'mean']].mean(axis=0), lvl2_train.loc[:, idx[:,'mean']].std(axis=0)

lvl2_train.loc[:, idx[:,'mean']] = (lvl2_train.loc[:, idx[:,'mean']] - lvl2_means)/lvl2_stds
lvl2_dev.loc[:, idx[:,'mean']] = (lvl2_dev.loc[:, idx[:,'mean']] - lvl2_means)/lvl2_stds
lvl2_test.loc[:, idx[:,'mean']] = (lvl2_test.loc[:, idx[:,'mean']] - lvl2_means)/lvl2_stds


KeyError: "None of [Index([slice(None, None, None), 'mean'], dtype='object')] are in the [columns]"